In [11]:
# Import Libraries and Modules
from pathlib import Path
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data" / "original" / "winequality-red.csv").exists())
sys.path.insert(0, str(project_root))

from app.data import load_training_data
from app.evaluation import evaluate_models
from app.predict import predict_quality
from models.random_forest import train_random_forest
from models.xgboost import train_xgboost
from models.support_vector_regression import train_support_vector_regression

In [12]:
# Data Preprocessing and Splitting
clean, x_train, x_test, y_train, y_test = load_training_data()
print(f"Cleaned rows: {len(clean)}; training: {len(x_train)}; held out: {len(x_test)}")

Cleaned rows: 1359; training: 1087; held out: 272


In [13]:
# Model Fitting
models = {
    "Random Forest": train_random_forest(x_train, y_train),
    "XGBoost": train_xgboost(x_train, y_train),
    "Support Vector Regression": train_support_vector_regression(x_train, y_train),
}
print("Trained:", ", ".join(models))

Trained: Random Forest, XGBoost, Support Vector Regression


In [14]:
# Five-Fold Cross-Validation on Training Rows
# Report the mean and standard deviation of MAE, RMSE, and R².
comparison = evaluate_models(models, x_train, y_train)
print(comparison.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

                    Model  MAE (points)  MAE SD  RMSE (points)  RMSE SD     R²  R² SD
            Random Forest        0.4882  0.0271         0.6444   0.0405 0.3773 0.0825
                  XGBoost        0.5023  0.0212         0.6576   0.0350 0.3520 0.0796
Support Vector Regression        0.6177  0.0314         0.7967   0.0399 0.0492 0.1120


In [15]:
# Example Prediction
# Use a training row so the held-out test set stays separate from this example.
model_name = "Support Vector Regression"
score, estimate = predict_quality(models[model_name], x_train.iloc[0].to_dict())
print(f"{model_name}: score {score}, estimated quality {estimate:.3f} points")

Support Vector Regression: score 5, estimated quality 5.303 points
